# Análise YouTube Top 100 Songs 2025
## Objetivo: Identificar as 25 músicas mais tocadas usando PySpark e SQL

### 1. Importação e Configuração do Spark

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc

# Criar sessão Spark
spark = SparkSession.builder \
    .appName("YouTube Top 100 Analysis") \
    .getOrCreate()

### 2. Carregamento dos Dados

In [16]:
# Carregar CSV (multiLine=True para campos com quebras de linha)
df = spark.read.csv("youtube-top-100-songs-2025.csv", header=True, inferSchema=True, multiLine=True, escape='"')

# Visualizar schema
df.printSchema()

root
 |-- title: string (nullable = true)
 |-- fulltitle: string (nullable = true)
 |-- description: string (nullable = true)
 |-- view_count: integer (nullable = true)
 |-- categories: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- duration_string: timestamp (nullable = true)
 |-- live_status: boolean (nullable = true)
 |-- thumbnail: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- channel_url: string (nullable = true)
 |-- channel_follower_count: integer (nullable = true)



### 3. Exploração Inicial

In [17]:
# Quantidade de registros
print(f"Total de músicas: {df.count()}")

# Amostra dos dados
df.select("title", "channel", "view_count").show(5, truncate=False)

Total de músicas: 100
+---------------------------------------------------------------+-------------+----------+
|title                                                          |channel      |view_count|
+---------------------------------------------------------------+-------------+----------+
|ROSÉ & Bruno Mars - APT. (Official Music Video)               |ROSÉ         |2009014557|
|Lady Gaga, Bruno Mars - Die With A Smile (Official Music Video)|Lady Gaga    |1324833300|
|Reneé Rapp - Leave Me Alone (Official Music Video)             |Reneé Rapp   |2536628   |
|Billie Eilish - BIRDS OF A FEATHER (Official Music Video)      |Billie Eilish|558329099 |
|Reneé Rapp - Mad (Official Music Video)                        |Reneé Rapp   |2113548   |
+---------------------------------------------------------------+-------------+----------+
only showing top 5 rows


### 4. Análise com SQL - Top 25 Músicas

In [18]:
# Registrar DataFrame como tabela temporária
df.createOrReplaceTempView("youtube_songs")

# Query SQL para top 25
top_25 = spark.sql("""
    SELECT 
        title,
        channel,
        CAST(view_count AS BIGINT) as view_count
    FROM youtube_songs
    ORDER BY CAST(view_count AS BIGINT) DESC
    LIMIT 25
""")

top_25.show(25, truncate=False)

+--------------------------------------------------------------------+-----------------+----------+
|title                                                               |channel          |view_count|
+--------------------------------------------------------------------+-----------------+----------+
|ROSÉ & Bruno Mars - APT. (Official Music Video)                    |ROSÉ             |2009014557|
|Lady Gaga, Bruno Mars - Die With A Smile (Official Music Video)     |Lady Gaga        |1324833300|
|Billie Eilish - BIRDS OF A FEATHER (Official Music Video)           |Billie Eilish    |558329099 |
|Sabrina Carpenter - Espresso                                        |Sabrina Carpenter|472570966 |
|Kendrick Lamar - Not Like Us                                        |Kendrick Lamar   |397228595 |
|Shaboozey - A Bar Song (Tipsy) [Official Visualizer]                |Shaboozey        |288277902 |
|Sabrina Carpenter - Please Please Please                            |Sabrina Carpenter|253618903 |


### 5. Exportar Resultado para TXT

In [20]:
# Coletar dados e salvar em TXT
top_25_list = top_25.collect()

with open("top_25_musicas_2025.txt", "w", encoding="utf-8") as f:
    f.write("TOP 25 MÚSICAS MAIS TOCADAS NO YOUTUBE - 2025\n")
    f.write("=" * 60 + "\n\n")
    
    for i, row in enumerate(top_25_list, 1):
        views = int(row['view_count']) if row['view_count'] else 0
        f.write(f"{i}. {row['title']}\n")
        f.write(f"   Canal: {row['channel']}\n")
        f.write(f"   Views: {views:,}\n\n")

print("Arquivo 'top_25_musicas.txt' criado com sucesso!")

Arquivo 'top_25_musicas.txt' criado com sucesso!


### 6. Encerrar Sessão Spark

In [14]:
spark.stop()